# Notebook 09: Evaluation Metrics

## Why This Matters

"Measure what matters" is easy to say and hard to do in recommender systems. Netflix discovered that a model with excellent offline RMSE didn't improve online engagement at all — leading them to shift focus from rating prediction to ranking quality. Spotify found that optimizing for stream count (online metric) led to shorter, more "sticky" songs — an unintended behavioral change in the catalog. Amazon has found that Recall@50 is a better predictor of purchase conversion than NDCG@10 for certain categories.

Understanding the full metric landscape — offline ranking metrics, catalog health metrics, and the gap between offline and online evaluation — is what separates a researcher's accuracy benchmark from a production system that actually serves users well.

## Background

### The Offline-Online Metric Gap

Offline metrics measure model performance on historical held-out data. Online metrics measure real user behavior in an A/B test. They often diverge because:

1. **Popularity bias in historical data**: Users mostly interact with popular items that were prominently displayed. Held-out test data over-represents popular items. A model that's great at predicting popular item preferences will score well offline but may not improve discovery.

2. **Feedback loops**: The recommender system *generates* the data it trains on. Items shown more often get more interactions, making them seem more popular, leading to more showing. This is a self-reinforcing bias.

3. **Position bias**: Users tend to click items in prominent positions regardless of quality. Historical data conflates item quality with display position. Models trained on this data learn position preferences, not item quality.

4. **Counterfactual problem**: We only observe user behavior on items that *were* shown. We can't measure what would have happened if we'd shown different items. Offline evaluation assumes missing items were disliked, which is wrong.

### Offline Metrics Taxonomy

**Rating prediction**: RMSE, MAE — useful for explicit feedback, poor proxy for ranking quality.

**Ranking accuracy**:
- $\text{Precision@K} = \frac{|\text{relevant items in top-K}|}{K}$
- $\text{Recall@K} = \frac{|\text{relevant items in top-K}|}{|\text{all relevant items}|}$
- $\text{F1@K} = \frac{2 \cdot P@K \cdot R@K}{P@K + R@K}$
- $\text{AP@K} = \frac{1}{\min(K, |\text{rel}|)} \sum_{k=1}^{K} P@k \cdot \text{rel}(k)$
- $\text{MAP@K}$ = mean AP@K over all users
- $\text{MRR} = \frac{1}{|U|} \sum_u \frac{1}{\text{rank of first relevant item}}$
- $\text{NDCG@K} = \frac{DCG@K}{IDCG@K}$ with $DCG@K = \sum_{k=1}^K \frac{2^{\text{rel}_k}-1}{\log_2(k+1)}$

**Catalog health**:
- **Coverage@K**: fraction of the item catalog recommended to at least one user
- **Novelty**: how often the recommended items are "surprising" (not just popular)
- **Diversity**: average pairwise distance within a user's recommendation list
- **Serendipity**: items that are surprising *and* relevant (hardest to compute)

### Online Metrics

CTR (click-through rate), conversion rate, session length, return rate, and long-term retention are the real signals. The trick is designing A/B tests with enough statistical power and the right guardrail metrics to detect improvements without causing harm.

**Interleaving**: A more sensitive A/B method for ranking — instead of splitting users into two groups, for each user interleave the results of two rankers and measure which items users prefer. Requires ~10x fewer users for the same statistical power as a standard A/B test.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn scipy tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings, os, requests, zipfile, io

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
movies = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1] + list(range(5,24)),
                     names=["item_id","title"] + [f"genre_{i}" for i in range(19)])
genre_cols = [f"genre_{i}" for i in range(19)]

n_users = ratings.user_id.max()
n_items = ratings.item_id.max()

# Temporal split
ratings_sorted = ratings.sort_values(["user_id", "timestamp"])
train_list, test_list = [], []
for uid, group in ratings_sorted.groupby("user_id"):
    n = len(group)
    cutoff = max(1, int(n * 0.8))
    train_list.append(group.iloc[:cutoff])
    test_list.append(group.iloc[cutoff:])
train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

user_pos_train = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
user_pos_test  = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
user_liked_test = (test_df[test_df.rating >= 4]
                   .groupby("user_id")["item_id"].apply(set).to_dict())

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train_df):,}, Test: {len(test_df):,}")

# Simulate two recommender models via simple popularity-based scoring
# Model A: popularity-based (biased, typical cold baseline)
item_popularity = train_df.groupby("item_id").size().reindex(range(1, n_items+1), fill_value=0)

# Model B: personalized (user-mean + item-mean, unbiased)
user_mean = train_df.groupby("user_id")["rating"].mean()
item_mean = train_df.groupby("item_id")["rating"].mean()
global_mean = train_df["rating"].mean()

print("Ready.")

## 1. Precision@K, Recall@K, F1@K

The foundational ranking metrics. These are threshold-independent: at each rank $K$, we evaluate the precision-recall tradeoff. In a recommendation carousel showing K items, Precision@K is "fraction of slots that showed relevant items" and Recall@K is "fraction of all things the user cares about that we found."

When to use which:
- **Precision@K**: when showing $K$ items and all slots are equally visible (e.g., a carousel of exactly K items)
- **Recall@K**: when the user will scroll through the list and you want to ensure relevant items appear somewhere in the top-K (e.g., search results)

In [ ]:
def precision_at_k(recs, relevant, K):
    """Precision@K: fraction of top-K that are relevant."""
    return len(set(recs[:K]) & relevant) / K

def recall_at_k(recs, relevant, K):
    """Recall@K: fraction of relevant items in top-K."""
    if not relevant:
        return 0.0
    return len(set(recs[:K]) & relevant) / len(relevant)

def f1_at_k(recs, relevant, K):
    p = precision_at_k(recs, relevant, K)
    r = recall_at_k(recs, relevant, K)
    return 2 * p * r / (p + r + 1e-10)

def average_precision_at_k(recs, relevant, K):
    """AP@K: average precision at each hit in top-K."""
    hits = 0
    ap = 0.0
    for k, item in enumerate(recs[:K], 1):
        if item in relevant:
            hits += 1
            ap += hits / k
    n_rel = min(len(relevant), K)
    return ap / n_rel if n_rel > 0 else 0.0

def ndcg_at_k(recs, relevant, K, use_graded=False, ratings_map=None):
    """NDCG@K: position-weighted precision."""
    dcg = 0.0
    for k, item in enumerate(recs[:K], 1):
        if item in relevant:
            if use_graded and ratings_map:
                rel = ratings_map.get(item, 1) - 1  # 0-based gain
            else:
                rel = 1
            dcg += (2**rel - 1) / np.log2(k + 1)

    # IDCG: ideal ranking
    if use_graded and ratings_map:
        ideal_rels = sorted([ratings_map.get(i, 1) - 1 for i in relevant], reverse=True)[:K]
    else:
        ideal_rels = [1] * min(len(relevant), K)
    idcg = sum((2**r - 1) / np.log2(k + 2) for k, r in enumerate(ideal_rels))

    return dcg / idcg if idcg > 0 else 0.0

def mrr(recs, relevant, K):
    """MRR: reciprocal rank of first hit."""
    for k, item in enumerate(recs[:K], 1):
        if item in relevant:
            return 1.0 / k
    return 0.0

def hit_rate_at_k(recs, relevant, K):
    """HR@K: 1 if any relevant item in top-K, else 0."""
    return 1 if len(set(recs[:K]) & relevant) > 0 else 0


# Quick demo on one user
sample_user = 1
recs_example = [3, 7, 25, 50, 10, 15, 100, 200, 300, 400]  # example ranked list
relevant_example = {3, 25, 100, 500}  # true positive items for this user

K = 10
print(f"Example recommendations: {recs_example}")
print(f"Relevant items: {relevant_example}")
print()
for k in [5, 10]:
    p = precision_at_k(recs_example, relevant_example, k)
    r = recall_at_k(recs_example, relevant_example, k)
    n = ndcg_at_k(recs_example, relevant_example, k)
    m = mrr(recs_example, relevant_example, k)
    ap = average_precision_at_k(recs_example, relevant_example, k)
    print(f"K={k}: P@K={p:.3f}  R@K={r:.3f}  NDCG@K={n:.3f}  MRR={m:.3f}  AP@K={ap:.3f}")

## 2. Full Evaluation Suite: Two Recommender Models

We compare a popularity-based baseline (Model A) against a simple personalized model (Model B). Popularity ranking is a surprisingly strong baseline — always recommend the most interacted items — but it's bad for diversity and fairness.

In [ ]:
def get_recommendations_popularity(user_id, item_pop, user_pos_train, K=20):
    """Model A: rank all items by global popularity, filter seen."""
    seen = user_pos_train.get(user_id, set())
    recs = [i for i in item_pop.sort_values(ascending=False).index if i not in seen][:K]
    return recs

def get_recommendations_personalized(user_id, train_df, user_mean, item_mean,
                                      global_mean, user_pos_train, K=20):
    """Model B: user-mean + item-mean - global mean (simple bias model)."""
    u_mean = user_mean.get(user_id, global_mean)
    seen = user_pos_train.get(user_id, set())

    scores = {}
    for item_id in range(1, n_items + 1):
        if item_id in seen:
            continue
        i_mean = item_mean.get(item_id, global_mean)
        scores[item_id] = u_mean + i_mean - global_mean

    return sorted(scores, key=scores.get, reverse=True)[:K]

# Evaluate both models
def evaluate_model(get_recs_fn, user_pos_test, user_liked_test, K_vals=[5, 10, 20], n_eval=500):
    """Compute all metrics for a recommendation function."""
    eval_users = [u for u in user_liked_test if len(user_liked_test[u]) > 0][:n_eval]
    results = {k: {"P": [], "R": [], "NDCG": [], "MRR": [], "AP": [], "HR": []} for k in K_vals}

    for u in eval_users:
        relevant = user_liked_test[u]
        recs = get_recs_fn(u)
        max_k = max(K_vals)
        for K in K_vals:
            results[K]["P"].append(precision_at_k(recs, relevant, K))
            results[K]["R"].append(recall_at_k(recs, relevant, K))
            results[K]["NDCG"].append(ndcg_at_k(recs, relevant, K))
            results[K]["MRR"].append(mrr(recs, relevant, K))
            results[K]["AP"].append(average_precision_at_k(recs, relevant, K))
            results[K]["HR"].append(hit_rate_at_k(recs, relevant, K))

    return {K: {m: np.mean(v) for m, v in metrics.items()} for K, metrics in results.items()}


import functools
pop_fn = functools.partial(get_recommendations_popularity,
                            item_pop=item_popularity, user_pos_train=user_pos_train)
pers_fn = functools.partial(get_recommendations_personalized,
                              train_df=train_df, user_mean=user_mean, item_mean=item_mean,
                              global_mean=global_mean, user_pos_train=user_pos_train)

print("Evaluating Model A (Popularity)...")
results_pop = evaluate_model(pop_fn, user_pos_test, user_liked_test)
print("Evaluating Model B (Personalized)...")
results_pers = evaluate_model(pers_fn, user_pos_test, user_liked_test)

# Print comparison table
K = 10
print(f"\n{'Metric':<10} {'Model A (Pop)':>15} {'Model B (Pers)':>15}")
print("-" * 42)
for metric in ["P", "R", "NDCG", "MRR", "AP", "HR"]:
    a_val = results_pop[K][metric]
    b_val = results_pers[K][metric]
    delta = (b_val - a_val) / (a_val + 1e-8) * 100
    print(f"{metric+'@'+str(K):<10} {a_val:>15.4f} {b_val:>15.4f}  (Δ={delta:+.1f}%)")

## 3. Precision-Recall Tradeoff Curve

As K increases, recall increases (more items, more chances to hit relevant ones) but precision typically decreases (more items shown, harder to maintain quality). The area under the PR curve summarizes model quality across all K values.

In [ ]:
K_range = list(range(1, 51))

def pr_curve(get_recs_fn, user_liked_test, K_range, n_eval=400):
    eval_users = [u for u in user_liked_test if len(user_liked_test[u]) > 0][:n_eval]
    precisions, recalls = [], []
    for K in K_range:
        p_list, r_list = [], []
        for u in eval_users:
            relevant = user_liked_test[u]
            recs = get_recs_fn(u)
            p_list.append(precision_at_k(recs, relevant, K))
            r_list.append(recall_at_k(recs, relevant, K))
        precisions.append(np.mean(p_list))
        recalls.append(np.mean(r_list))
    return precisions, recalls


p_pop, r_pop = pr_curve(pop_fn, user_liked_test, K_range)
p_pers, r_pers = pr_curve(pers_fn, user_liked_test, K_range)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# NDCG@K curve
ndcg_pop_k  = [np.mean([ndcg_at_k(pop_fn(u),  user_liked_test[u], K)
                         for u in list(user_liked_test.keys())[:300] if user_liked_test[u]])
               for K in [1,2,5,10,20,50]]
ndcg_pers_k = [np.mean([ndcg_at_k(pers_fn(u), user_liked_test[u], K)
                         for u in list(user_liked_test.keys())[:300] if user_liked_test[u]])
               for K in [1,2,5,10,20,50]]

axes[0].plot([1,2,5,10,20,50], ndcg_pop_k, marker="o", label="Popularity", color="coral")
axes[0].plot([1,2,5,10,20,50], ndcg_pers_k, marker="s", label="Personalized", color="steelblue")
axes[0].set_xlabel("K")
axes[0].set_ylabel("NDCG@K")
axes[0].set_title("NDCG@K vs K")
axes[0].legend()
axes[0].set_xscale("log")

# P@K and R@K curves
axes[1].plot(K_range, p_pop, color="coral", alpha=0.8, label="Pop P@K")
axes[1].plot(K_range, p_pers, color="steelblue", alpha=0.8, label="Pers P@K")
axes[1].plot(K_range, r_pop, color="coral", linestyle="--", alpha=0.8, label="Pop R@K")
axes[1].plot(K_range, r_pers, color="steelblue", linestyle="--", alpha=0.8, label="Pers R@K")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")
axes[1].set_title("Precision & Recall vs K")
axes[1].legend(fontsize=8)

# PR Curve
axes[2].plot(r_pop, p_pop, color="coral", label="Popularity", linewidth=2)
axes[2].plot(r_pers, p_pers, color="steelblue", label="Personalized", linewidth=2)
axes[2].set_xlabel("Recall@K")
axes[2].set_ylabel("Precision@K")
axes[2].set_title("Precision-Recall Curve\n(K varies from 1 to 50)")
axes[2].legend()

plt.tight_layout()
plt.savefig("/tmp/pr_curves.png", dpi=120)
plt.show()

## 4. Catalog Health Metrics: Coverage, Novelty, Diversity

A model that recommends the same 100 popular items to every user may score well on NDCG@10 but fails on fairness (long-tail items never get shown), diversity (users see the same things), and discovery (no serendipitous finds).

In [ ]:
def compute_catalog_metrics(get_recs_fn, item_pop, movies_df,
                             genre_cols, n_users_eval=500, K=10):
    """Coverage, Novelty, and Diversity metrics."""
    all_recs = {}
    for u in range(1, min(n_users_eval+1, n_users+1)):
        all_recs[u] = get_recs_fn(u)[:K]

    all_recommended = set(item for recs in all_recs.values() for item in recs)

    # Coverage: fraction of catalog recommended
    coverage = len(all_recommended) / n_items

    # Novelty: how surprising are the recommendations?
    # Define: log(1 / popularity_rank) — popular items are less novel
    pop_rank = item_pop.rank(ascending=False)
    max_rank = len(pop_rank)
    novelty_scores = []
    for recs in all_recs.values():
        user_novelty = np.mean([np.log2(1 + pop_rank.get(i, max_rank) / max_rank * 100 + 1)
                                 for i in recs if i in pop_rank.index])
        novelty_scores.append(user_novelty)
    novelty = np.mean(novelty_scores)

    # Diversity: intra-list diversity = average pairwise distance in genre space
    genre_matrix = movies_df.set_index("item_id")[genre_cols].fillna(0)
    diversity_scores = []
    for recs in all_recs.values():
        valid_recs = [r for r in recs if r in genre_matrix.index]
        if len(valid_recs) < 2:
            continue
        embs = genre_matrix.loc[valid_recs].values.astype(float)
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        embs_norm = embs / np.clip(norms, 1e-8, None)
        sims = cosine_similarity(embs_norm)
        n = len(valid_recs)
        avg_sim = (sims.sum() - n) / (n * (n-1)) if n > 1 else 0
        diversity_scores.append(1 - avg_sim)
    diversity = np.mean(diversity_scores)

    return {
        "Coverage@K": coverage,
        "Novelty": novelty,
        "Diversity": diversity,
        "Unique items recommended": len(all_recommended),
    }


metrics_a = compute_catalog_metrics(pop_fn, item_popularity, movies, genre_cols, K=10)
metrics_b = compute_catalog_metrics(pers_fn, item_popularity, movies, genre_cols, K=10)

print(f"{'Metric':<30} {'Popularity':>12} {'Personalized':>14}")
print("-" * 58)
for key in metrics_a:
    a_val = metrics_a[key]
    b_val = metrics_b[key]
    fmt = ".3f" if isinstance(a_val, float) else ",d"
    print(f"{key:<30} {a_val:>12.3f} {b_val:>14.3f}")

# Visualize the coverage difference
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pop_item_exposure = {i: 0 for i in range(1, n_items+1)}
pers_item_exposure = {i: 0 for i in range(1, n_items+1)}

for u in range(1, 501):
    for item in pop_fn(u)[:10]:
        pop_item_exposure[item] = pop_item_exposure.get(item, 0) + 1
    for item in pers_fn(u)[:10]:
        pers_item_exposure[item] = pers_item_exposure.get(item, 0) + 1

sorted_pop_exp  = sorted(pop_item_exposure.values(), reverse=True)
sorted_pers_exp = sorted(pers_item_exposure.values(), reverse=True)

axes[0].plot(sorted_pop_exp, color="coral", label="Popularity", linewidth=2)
axes[0].plot(sorted_pers_exp, color="steelblue", label="Personalized", linewidth=2)
axes[0].set_xlabel("Item rank (by exposure)")
axes[0].set_ylabel("Times recommended (500 users)")
axes[0].set_title("Item Exposure Distribution\nPopularity = extreme concentration")
axes[0].legend()
axes[0].set_yscale("log")

# Genre distribution of recommended items
genre_names_vis = ["Action","Comedy","Drama","Romance","Sci-Fi","Horror","Thriller"]
genre_idx = [1, 5, 8, 14, 15, 11, 16]

def genre_distribution(get_recs_fn, movies_df, genre_cols, n_users=300, K=10):
    from collections import Counter
    genre_counts = Counter()
    for u in range(1, n_users+1):
        recs = get_recs_fn(u)[:K]
        for item in recs:
            row = movies_df[movies_df.item_id == item]
            if len(row):
                for g_col in genre_cols:
                    if row.iloc[0][g_col] == 1:
                        genre_counts[g_col] += 1
    return genre_counts

gc_pop  = genre_distribution(pop_fn, movies, genre_cols)
gc_pers = genre_distribution(pers_fn, movies, genre_cols)

x = np.arange(len(genre_names_vis))
w = 0.35
axes[1].bar(x - w/2, [gc_pop.get(genre_cols[i], 0) for i in genre_idx],
            w, label="Popularity", color="coral", alpha=0.8)
axes[1].bar(x + w/2, [gc_pers.get(genre_cols[i], 0) for i in genre_idx],
            w, label="Personalized", color="steelblue", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(genre_names_vis, rotation=30, ha="right")
axes[1].set_title("Genre Distribution in Recommendations")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/catalog_metrics.png", dpi=120)
plt.show()

## 5. The Offline-Online Metric Gap

This simulation demonstrates why optimizing for offline NDCG doesn't guarantee online CTR improvement. Popularity bias in offline data creates a false ceiling.

In [ ]:
# Demonstrate offline-online divergence via simulation
# Simulate a world where item quality != popularity (the common case)
np.random.seed(123)

n_sim_items = 100
# True item qualities (unknown in offline data)
true_quality = np.random.beta(2, 3, n_sim_items)

# Popularity (highly correlated with quality, but with noise + rich-get-richer bias)
popularity_bias = np.cumsum(np.random.exponential(1, n_sim_items))[::-1]
popularity_bias /= popularity_bias.max()
# Offline "apparent quality" = true quality + popularity bias
offline_quality = 0.4 * true_quality + 0.6 * popularity_bias + np.random.normal(0, 0.05, n_sim_items)
offline_quality = np.clip(offline_quality, 0, 1)

# Model A: optimize offline quality
rank_offline = np.argsort(offline_quality)[::-1]
# Model B: optimize true quality (oracle)
rank_oracle = np.argsort(true_quality)[::-1]
# Model C: random baseline
rank_random = np.random.permutation(n_sim_items)

# Compute online CTR@10 for each model (true quality determines click probability)
def online_ctr(ranking, true_quality, K=10, n_impressions=10000):
    """Simulate online A/B test: users click based on true quality."""
    top_k = ranking[:K]
    click_probs = true_quality[top_k]
    return click_probs.mean()

def offline_ndcg(ranking, offline_quality, K=10):
    """Offline NDCG using offline quality scores as relevance."""
    rels = offline_quality[ranking[:K]]
    dcg = sum(r / np.log2(i+2) for i, r in enumerate(rels))
    ideal_rels = np.sort(offline_quality)[::-1][:K]
    idcg = sum(r / np.log2(i+2) for i, r in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0

models = {"Offline-optimized (Model A)": rank_offline,
          "Oracle (true quality)": rank_oracle,
          "Random baseline": rank_random}

print(f"{'Model':<35} {'Offline NDCG@10':>17} {'Online CTR@10 (true)':>20}")
print("-" * 74)
for name, ranking in models.items():
    off_ndcg = offline_ndcg(ranking, offline_quality)
    on_ctr   = online_ctr(ranking, true_quality)
    print(f"{name:<35} {off_ndcg:>17.4f} {on_ctr:>20.4f}")

print("\nKey insight: offline-optimized model has best offline NDCG but")
print("does NOT necessarily have best online CTR — popularity bias inflates offline scores.")

# Scatter: offline NDCG vs true quality for individual items
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(true_quality, offline_quality, alpha=0.5, c="steelblue", s=30)
corr = np.corrcoef(true_quality, offline_quality)[0, 1]
axes[0].set_xlabel("True Item Quality (online CTR)")
axes[0].set_ylabel("Offline Apparent Quality (popularity-biased)")
axes[0].set_title(f"Offline vs True Quality\nSpearman r={corr:.2f} — biased by popularity")

# Position bias illustration
position_bias = np.array([1.0, 0.72, 0.55, 0.42, 0.30, 0.22, 0.15, 0.10, 0.07, 0.05])
axes[1].bar(range(1, 11), position_bias, color="coral", edgecolor="white")
axes[1].set_xlabel("Recommendation Position")
axes[1].set_ylabel("Relative Click Rate")
axes[1].set_title("Position Bias: Items at Position 1\nget ~20x more clicks than Position 10")

plt.tight_layout()
plt.savefig("/tmp/offline_online_gap.png", dpi=120)
plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Precision@K | Fraction of top-K that are relevant — measures recommendation quality |
| Recall@K | Fraction of relevant items found in top-K — measures coverage |
| NDCG@K | Position-weighted — hitting relevant items early is rewarded |
| MRR | Reciprocal rank of first relevant item — good for single-slot scenarios |
| MAP@K | Mean Average Precision — summarizes precision at every hit position |
| Coverage@K | Fraction of catalog shown to any user — captures long-tail fairness |
| Novelty | Inverse popularity of recommendations — measures discovery potential |
| Diversity | Intra-list dissimilarity — prevents boring, repetitive slates |
| Offline-online gap | Popularity bias inflates offline metrics — online A/B test is ground truth |
| Position bias | Items shown first get more clicks regardless of quality — debiasing required |

### Common Pitfalls
- Optimizing only NDCG@10 — ignores coverage, novelty, diversity, and long-tail fairness
- Treating offline NDCG as a proxy for online CTR — they diverge due to popularity bias
- Not controlling for position bias in evaluation — conflates display order with item quality
- Using the same test set to tune and evaluate — leads to overfitting on evaluation data
- Comparing models on different user subsets — always use the same eval population